##Bonus (Optional)- A Particle Simulation Application
This exercise will look into a simplified version of a particle-in-cell simulation code and focus on its particle mover that calculates and updates the position of particles.

In [1]:
!git clone https://github.com/KTH-ScaLab/DD2360-HT25.git

Cloning into 'DD2360-HT25'...
remote: Enumerating objects: 807, done.
remote: Counting objects: 100% (807/807), done.
remote: Compressing objects: 100% (611/611), done.
remote: Total 807 (delta 158), reused 807 (delta 158), pack-reused 0 (from 0)
Receiving objects: 100% (807/807), 22.35 MiB | 18.56 MiB/s, done.
Resolving deltas: 100% (158/158), done.


### New Makefile

In [ ]:
# From https://x.momo86.net/?p=29

CXX=g++
CXXFLAGS=-std=c++11 -I./include -O3 -g -Xcompiler -Wall

NVCC=nvcc
ARCH=sm_75
NVCCFLAGS= -I./include -arch=$(ARCH) -std=c++11 -O3 -g -Xcompiler -Wall --compiler-bindir=$(CXX)

SRCDIR=src
SRCS=$(shell find $(SRCDIR) -name '*.cu' -o -name '*.cpp')

OBJDIR=src
OBJS=$(subst $(SRCDIR),$(OBJDIR), $(SRCS))
OBJS:=$(subst .cpp,.o,$(OBJS))
OBJS:=$(subst .cu,.o,$(OBJS))

BIN := ./bin
TARGET=miniPIC.out

all: dir $(BIN)/$(TARGET)

dir: ${BIN}

${BIN}:
	mkdir -p $(BIN)

$(BIN)/$(TARGET): $(OBJS)
	$(NVCC) $(NVCCFLAGS) $+ -o $@

$(SRCDIR)/%.o: $(SRCDIR)/%.cu
	$(NVCC) $(NVCCFLAGS) $< -c -o $@

$(OBJDIR)/%.o: $(SRCDIR)/%.cpp
	[ -d $(OBJDIR) ] || mkdir $(OBJDIR)
	$(NVCC) $(CXXFLAGS) $< -c -o $@

clean:
	rm -rf $(OBJS)
	rm -rf $(TARGET)


Test Run

In [2]:
import os
os.chdir('DD2360-HT25/iPIC3D-mini')

In [3]:
!pwd

/content/DD2360-HT25/iPIC3D-mini


In [7]:
!make clean
!make


rm -rf  src/ConfigFile.o src/Timing.o src/EMfield_aux.o src/Grid.o src/sputniPIC.o src/BC.o src/InterpDensSpecies.o src/TransArraySpace3D.o src/Particles_aux.o src/Particles.o src/Basic.o src/EMfield.o src/InterpDensNet.o src/RW_IO.o
rm -rf miniPIC.out
[ -d src ] || mkdir src
nvcc -std=c++11 -I./include -O3 -g -Xcompiler -Wall src/ConfigFile.cpp -c -o src/ConfigFile.o
[ -d src ] || mkdir src
nvcc -std=c++11 -I./include -O3 -g -Xcompiler -Wall src/Timing.cpp -c -o src/Timing.o
[ -d src ] || mkdir src
nvcc -std=c++11 -I./include -O3 -g -Xcompiler -Wall src/EMfield_aux.cpp -c -o src/EMfield_aux.o
[ -d src ] || mkdir src
nvcc -std=c++11 -I./include -O3 -g -Xcompiler -Wall src/Grid.cpp -c -o src/Grid.o
[ -d src ] || mkdir src
nvcc -std=c++11 -I./include -O3 -g -Xcompiler -Wall src/sputniPIC.cpp -c -o src/sputniPIC.o
In file included from src/sputniPIC.cpp:25:
./include/IC.h: In function ‘void initGEM(parameters*, grid*, EMfield*, EMfield_aux*, particles*, interpDensSpecies*)’:
./include/IC.

CPU Run

In [ ]:
!./bin/miniPIC.out inputfiles/GEM_2D.inp


-------------------------
Simulation Parameters
-------------------------
Number of species    = 4
Number of particles of species 0 = 4096000	 (MAX = 4096000)  QOM = -64
Number of particles of species 1 = 4096000	 (MAX = 4096000)  QOM = 1
Number of particles of species 2 = 4096000	 (MAX = 4096000)  QOM = -64
Number of particles of species 3 = 4096000	 (MAX = 4096000)  QOM = 1
x-Length                 = 40
y-Length                 = 20
z-Length                 = 1
Number of cells (x)      = 256
Number of cells (y)      = 128
Number of cells (z)      = 1
Time step                = 0.25
Number of cycles         = 10
Results saved in: data
*************************************************
**  Initialize GEM Challenge with Pertubation  **
*************************************************
** B0x = 0.0195
** B0y = 0
** B0z = 0
** Delta (current sheet thickness) = 0.5
** rho species 0 = 1 CURRENT SHEET 
** rho species 1 = 1 CURRENT SHEET 
** rho species 2 = 0.02 BACKGROUND 
** rho species 3 =

Code to add to Particles.cu

In [ ]:
__global__ void mover_pc_kernel(
    long nop, FPpart dt_sub_cycling, FPpart dto2, FPpart qom, FPpart c,
    int n_sub_cycles, int NiterMover,
    FPpart* d_x, FPpart* d_y, FPpart* d_z,
    FPpart* d_u, FPpart* d_v, FPpart* d_w,
    // Grid parameters & flat coordinates
    FPpart xStart, FPpart yStart, FPpart zStart,
    FPpart invdx, FPpart invdy, FPpart invdz, FPpart invVOL,
    FPpart Lx, FPpart Ly, FPpart Lz,
    int nxn, int nyn, int nzn,
    const FPfield* __restrict__ XN_flat,
    const FPfield* __restrict__ YN_flat,
    const FPfield* __restrict__ ZN_flat,
    const FPfield* __restrict__ Ex_flat,
    const FPfield* __restrict__ Ey_flat,
    const FPfield* __restrict__ Ez_flat,
    const FPfield* __restrict__ Bxn_flat,
    const FPfield* __restrict__ Byn_flat,
    const FPfield* __restrict__ Bzn_flat,
    bool PERIODICX, bool PERIODICY, bool PERIODICZ
) {

    long i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i >= nop) return;

    FPpart qomdt2 = qom * dto2 / c;

    // Subcycling loop
    for (int i_sub = 0; i_sub < n_sub_cycles; i_sub++) {

        FPpart xptilde = d_x[i];
        FPpart yptilde = d_y[i];
        FPpart zptilde = d_z[i];
        FPpart u = d_u[i];
        FPpart v = d_v[i];
        FPpart w = d_w[i];
        FPpart uptilde = u;
        FPpart vptilde = v;
        FPpart wptilde = w;

        for (int innter = 0; innter < NiterMover; innter++) {

            // Cell indices
            int ix = 2 + int((d_x[i] - xStart) * invdx);
            int iy = 2 + int((d_y[i] - yStart) * invdy);
            int iz = 2 + int((d_z[i] - zStart) * invdz);

            // Weights and interpolation using flat array indexing (stride = nyn*nzn and nzn)
            // Stride calculation matching Alloc.h: get_idx(x, y, z, nyn, nzn) = nyn*nzn*x + nzn*y + z
            long idx000 = get_idx(ix, iy, iz, nyn, nzn);

            FPpart xi[2], eta[2], zeta[2];
            xi[0]   = d_x[i] - XN_flat[ get_idx(ix - 1, iy,     iz,     nyn, nzn) ];
            eta[0]  = d_y[i] - YN_flat[ get_idx(ix,     iy - 1, iz,     nyn, nzn) ];
            zeta[0] = d_z[i] - ZN_flat[ get_idx(ix,     iy,     iz - 1, nyn, nzn) ];
            xi[1]   = XN_flat[ idx000 ] - d_x[i];
            eta[1]  = YN_flat[ idx000 ] - d_y[i];
            zeta[1] = ZN_flat[ idx000 ] - d_z[i];

            FPpart weight[2][2][2];
            for (int ii = 0; ii < 2; ii++)
                for (int jj = 0; jj < 2; jj++)
                    for (int kk = 0; kk < 2; kk++)
                        weight[ii][jj][kk] = xi[ii] * eta[jj] * zeta[kk] * invVOL;

            FPfield Exl = 0.0, Eyl = 0.0, Ezl = 0.0, Bxl = 0.0, Byl = 0.0, Bzl = 0.0;

            for (int ii = 0; ii < 2; ii++) {
                for (int jj = 0; jj < 2; jj++) {
                    for (int kk = 0; kk < 2; kk++) {
                        long field_idx = get_idx(ix - ii, iy - jj, iz - kk, nyn, nzn);
                        Exl += weight[ii][jj][kk] * Ex_flat[field_idx];
                        Eyl += weight[ii][jj][kk] * Ey_flat[field_idx];
                        Ezl += weight[ii][jj][kk] * Ez_flat[field_idx];
                        Bxl += weight[ii][jj][kk] * Bxn_flat[field_idx];
                        Byl += weight[ii][jj][kk] * Byn_flat[field_idx];
                        Bzl += weight[ii][jj][kk] * Bzn_flat[field_idx];
                    }
                }
            }

            // 3. Predictor-Corrector equations
            FPpart omdtsq = qomdt2 * qomdt2 * (Bxl * Bxl + Byl * Byl + Bzl * Bzl);
            FPpart denom = 1.0 / (1.0 + omdtsq);

            FPpart ut = u + qomdt2 * Exl;
            FPpart vt = v + qomdt2 * Eyl;
            FPpart wt = w + qomdt2 * Ezl;
            FPpart udotb = ut * Bxl + vt * Byl + wt * Bzl;

            uptilde = (ut + qomdt2 * (vt * Bzl - wt * Byl + qomdt2 * udotb * Bxl)) * denom;
            vptilde = (vt + qomdt2 * (wt * Bxl - ut * Bzl + qomdt2 * udotb * Byl)) * denom;
            wptilde = (wt + qomdt2 * (ut * Byl - vt * Bxl + qomdt2 * udotb * Bzl)) * denom;

            d_x[i] = xptilde + uptilde * dto2;
            d_y[i] = yptilde + vptilde * dto2;
            d_z[i] = zptilde + wptilde * dto2;
        }

        // Final velocity and position update
        d_u[i] = 2.0 * uptilde - u;
        d_v[i] = 2.0 * vptilde - v;
        d_w[i] = 2.0 * wptilde - w;
        d_x[i] = xptilde + uptilde * dt_sub_cycling;
        d_y[i] = yptilde + vptilde * dt_sub_cycling;
        d_z[i] = zptilde + wptilde * dt_sub_cycling;

        //////////
        //////////
        ////////// BC

        // X-DIRECTION
        if (d_x[i] > Lx) {
            if (PERIODICX == true) {
                d_x[i] = d_x[i] - Lx;
            } else {
                d_u[i] = -d_u[i];
                d_x[i] = 2 * Lx - d_x[i];
            }
        }
        if (d_x[i] < 0) {
            if (PERIODICX == true) {
                d_x[i] = d_x[i] + Lx;
            } else {
                d_u[i] = -d_u[i];
                d_x[i] = -d_x[i];
            }
        }

        // Y-DIRECTION
        if (d_y[i] > Ly) {
            if (PERIODICY == true) {
                d_y[i] = d_y[i] - Ly;
            }
            else {
                d_v[i] = -d_v[i];
                d_y[i] = 2 * Ly - d_y[i];
            }
        }
        if (d_y[i] < 0) {
            if (PERIODICY == true) {
                d_y[i] = d_y[i] + Ly;
            } else {
                d_v[i] = -d_v[i];
                d_y[i] = -d_y[i];
            }
        }

        // Z-DIRECTION
        if (d_z[i] > Lz) {
            if (PERIODICZ == true) {
                d_z[i] = d_z[i] - Lz;
            } else {
                d_w[i] = -d_w[i];
                d_z[i] = 2 * Lz - d_z[i];
            }
        }
        if (d_z[i] < 0) {
            if (PERIODICZ == true) {
                d_z[i] = d_z[i] + Lz;
            } else {
                d_w[i] = -d_w[i];
                d_z[i] = -d_z[i];
            }
        }
    }
}


int mover_PC_gpu(struct particles* part, struct EMfield* field, struct grid* grd, struct parameters* param)
{
    // Print species and subcycling info
    std::cout << "*** GPU MOVER with SUBCYCLYING " << param->n_sub_cycles << " - species " << part->species_ID << " ***" << std::endl;

    long nop = part->nop;
    long npmax = part->npmax;
    size_t bytes_part = npmax * sizeof(FPpart);

    // Allocate Device Memory for Particles
    FPpart *d_x, *d_y, *d_z, *d_u, *d_v, *d_w;
    cudaMalloc(&d_x, bytes_part);
    cudaMalloc(&d_y, bytes_part);
    cudaMalloc(&d_z, bytes_part);
    cudaMalloc(&d_u, bytes_part);
    cudaMalloc(&d_v, bytes_part);
    cudaMalloc(&d_w, bytes_part);

    // Copy Host Particles to Device
    cudaMemcpy(d_x, part->x, bytes_part, cudaMemcpyHostToDevice);
    cudaMemcpy(d_y, part->y, bytes_part, cudaMemcpyHostToDevice);
    cudaMemcpy(d_z, part->z, bytes_part, cudaMemcpyHostToDevice);
    cudaMemcpy(d_u, part->u, bytes_part, cudaMemcpyHostToDevice);
    cudaMemcpy(d_v, part->v, bytes_part, cudaMemcpyHostToDevice);
    cudaMemcpy(d_w, part->w, bytes_part, cudaMemcpyHostToDevice);

    // Allocate and Copy Grid & Field Data to Device
    size_t bytes_grid = (size_t)grd->nxn * grd->nyn * grd->nzn * sizeof(FPfield);

    FPfield *d_XN, *d_YN, *d_ZN;
    FPfield *d_Ex, *d_Ey, *d_Ez, *d_Bxn, *d_Byn, *d_Bzn;

    cudaMalloc(&d_XN, bytes_grid);
    cudaMalloc(&d_YN, bytes_grid);
    cudaMalloc(&d_ZN, bytes_grid);
    cudaMalloc(&d_Ex, bytes_grid);
    cudaMalloc(&d_Ey, bytes_grid);
    cudaMalloc(&d_Ez, bytes_grid);
    cudaMalloc(&d_Bxn, bytes_grid);
    cudaMalloc(&d_Byn, bytes_grid);
    cudaMalloc(&d_Bzn, bytes_grid);

    // Copy grid nodes and fields from host flat pointers to device
    cudaMemcpy(d_XN, grd->XN_flat, bytes_grid, cudaMemcpyHostToDevice);
    cudaMemcpy(d_YN, grd->YN_flat, bytes_grid, cudaMemcpyHostToDevice);
    cudaMemcpy(d_ZN, grd->ZN_flat, bytes_grid, cudaMemcpyHostToDevice);

    // field->Ex[0][0] points to the base of the contiguous 1D array created by newArr3
    cudaMemcpy(d_Ex, field->Ex[0][0], bytes_grid, cudaMemcpyHostToDevice);
    cudaMemcpy(d_Ey, field->Ey[0][0], bytes_grid, cudaMemcpyHostToDevice);
    cudaMemcpy(d_Ez, field->Ez[0][0], bytes_grid, cudaMemcpyHostToDevice);
    cudaMemcpy(d_Bxn, field->Bxn[0][0], bytes_grid, cudaMemcpyHostToDevice);
    cudaMemcpy(d_Byn, field->Byn[0][0], bytes_grid, cudaMemcpyHostToDevice);
    cudaMemcpy(d_Bzn, field->Bzn[0][0], bytes_grid, cudaMemcpyHostToDevice);

    // Set up execution configuration (threads and blocks)
    int threadsPerBlock = 256;
    int blocksPerGrid = (nop + threadsPerBlock - 1) / threadsPerBlock;

    FPpart dt_sub_cycling = (FPpart) param->dt / ((double) part->n_sub_cycles);
    FPpart dto2 = .5 * dt_sub_cycling;

    // Launch Kernel
    mover_pc_kernel<<<blocksPerGrid, threadsPerBlock>>>(
        nop, dt_sub_cycling, dto2, part->qom, (FPpart)param->c,
        part->n_sub_cycles, part->NiterMover,
        d_x, d_y, d_z, d_u, d_v, d_w,
        grd->xStart, grd->yStart, grd->zStart,
        grd->invdx, grd->invdy, grd->invdz, grd->invVOL,
        grd->Lx, grd->Ly, grd->Lz,
        grd->nxn, grd->nyn, grd->nzn,
        d_XN, d_YN, d_ZN,
        d_Ex, d_Ey, d_Ez, d_Bxn, d_Byn, d_Bzn,
        grd->PERIODICX, grd->PERIODICY, grd->PERIODICZ
    );

    // Wait for GPU to finish and check for launch errors
    cudaDeviceSynchronize();

    // Copy Updated Particles Back to Host
    cudaMemcpy(part->x, d_x, bytes_part, cudaMemcpyDeviceToHost);
    cudaMemcpy(part->y, d_y, bytes_part, cudaMemcpyDeviceToHost);
    cudaMemcpy(part->z, d_z, bytes_part, cudaMemcpyDeviceToHost);
    cudaMemcpy(part->u, d_u, bytes_part, cudaMemcpyDeviceToHost);
    cudaMemcpy(part->v, d_v, bytes_part, cudaMemcpyDeviceToHost);
    cudaMemcpy(part->w, d_w, bytes_part, cudaMemcpyDeviceToHost);

    // Cleanup Device Memory
    cudaFree(d_x); cudaFree(d_y); cudaFree(d_z);
    cudaFree(d_u); cudaFree(d_v); cudaFree(d_w);
    cudaFree(d_XN); cudaFree(d_YN); cudaFree(d_ZN);
    cudaFree(d_Ex); cudaFree(d_Ey); cudaFree(d_Ez);
    cudaFree(d_Bxn); cudaFree(d_Byn); cudaFree(d_Bzn);

    return 0;
}

GPU Run

In [8]:
!./bin/miniPIC.out inputfiles/GEM_2D.inp


-------------------------
Simulation Parameters
-------------------------
Number of species    = 4
Number of particles of species 0 = 4096000	 (MAX = 4096000)  QOM = -64
Number of particles of species 1 = 4096000	 (MAX = 4096000)  QOM = 1
Number of particles of species 2 = 4096000	 (MAX = 4096000)  QOM = -64
Number of particles of species 3 = 4096000	 (MAX = 4096000)  QOM = 1
x-Length                 = 40
y-Length                 = 20
z-Length                 = 1
Number of cells (x)      = 256
Number of cells (y)      = 128
Number of cells (z)      = 1
Time step                = 0.25
Number of cycles         = 10
Results saved in: data
*************************************************
**  Initialize GEM Challenge with Pertubation  **
*************************************************
** B0x = 0.0195
** B0y = 0
** B0z = 0
** Delta (current sheet thickness) = 0.5
** rho species 0 = 1 CURRENT SHEET 
** rho species 1 = 1 CURRENT SHEET 
** rho species 2 = 0.02 BACKGROUND 
** rho species 3 =